# 01 · 影像即張量：torchvision 的入口

歡迎來到 **電腦視覺 → 深度電腦視覺**。

這條軌道**假設你已經學過 `ml/pytorch`**(會 tensor、autograd、訓練迴圈、CNN 基礎)。我們不重教這些,而是直接做**視覺專屬的進階主題**:遷移學習、資料增強、物件偵測、影像分割、可解釋性。

第一步,先把最根本的觀念釘牢:

> **在電腦眼裡,一張彩色影像就是一個形狀 `[C, H, W]` 的張量**——3 個顏色通道(RGB)× 高 × 寬,每個數字是一個像素強度。所有視覺模型,吃的都是這個張量。

## 1. 安裝 torchvision

torchvision 提供資料集、預訓練模型與影像轉換工具。

In [ ]:
%pip install -q -U torchvision

In [ ]:
# CIFAR-10 的 10 個類別，以及這個資料集慣用的正規化平均/標準差。
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

## 2. 載入 CIFAR-10,看一張影像的「形狀」

CIFAR-10 是 6 萬張 32×32 的彩色小圖,分 10 類。`ToTensor()` 會把 PIL 影像轉成 `[3,32,32]` 的張量,並把像素從 0–255 縮到 **0–1**。

In [ ]:
import torchvision
from torchvision import transforms

train = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transforms.ToTensor()
)
img, label = train[0]
print("一張影像的形狀:", tuple(img.shape))      # (3, 32, 32) = (通道, 高, 寬)
print("像素範圍:", round(img.min().item(), 3), "~", round(img.max().item(), 3))
print("標籤:", label, "→", CIFAR10_CLASSES[label])

## 3. 把一批影像畫出來

用 `DataLoader` 一次拿一批,再用小工具排成 grid 看看。

In [ ]:
import matplotlib.pyplot as plt
import torch


def show_images(imgs, labels=None, classes=None, cols=8, denorm=None):
    """把一批張量影像 [N,C,H,W] 畫成 grid。

    denorm=(mean,std) 時會先反正規化回 0~1 再顯示（不然正規化過的圖會怪怪的）。
    """
    imgs = imgs.detach().cpu()
    n = len(imgs)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 1.3, rows * 1.45))
    for i in range(n):
        img = imgs[i]
        if denorm is not None:
            mean, std = denorm
            img = img * torch.tensor(std).view(-1, 1, 1) + torch.tensor(mean).view(-1, 1, 1)
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img)
        ax.axis("off")
        if labels is not None:
            lab = labels[i].item() if hasattr(labels[i], "item") else labels[i]
            ax.set_title(classes[lab] if classes else str(lab), fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(train, batch_size=16, shuffle=True)
imgs, labels = next(iter(loader))
print("一個 batch 的形狀:", tuple(imgs.shape))   # (16, 3, 32, 32)
show_images(imgs, labels, CIFAR10_CLASSES, cols=8)

## 4. 正規化:模型更好訓練的關鍵前處理

把每個通道減掉平均、除以標準差,讓輸入分布更集中,訓練更穩、更快。這是視覺模型幾乎必做的一步。

In [ ]:
import torch

norm_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
norm_train = torchvision.datasets.CIFAR10(root="./data", train=True, transform=norm_tf)
nimg, _ = norm_train[0]
print("正規化後像素範圍:", round(nimg.min().item(), 2), "~", round(nimg.max().item(), 2))
print("→ 不再是 0~1,而是以 0 為中心、有正有負")

## 小結

- **影像 = `[C,H,W]` 張量**:RGB 三通道 × 高 × 寬,每格一個像素值。
- `torchvision.datasets` 一行下載資料集,`transforms` 組裝前處理管線。
- `ToTensor()` 把像素縮到 0–1;`Normalize()` 再以平均/標準差置中,讓訓練更穩。

下一課:在這個彩色資料集上,訓練一個比 pytorch 軌道更實際的 **CNN**。